# Prova Técnica — Desenvolvedor Backend

## Parte 1 — Questão Teórica: Arquitetura de Drones

### Visão geral da arquitetura

Essa arquitetura pode ser entendida como uma estrutura distribuída para disponibilizar serviços relacionados a drones por meio de uma API centralizada.

Quando um usuário deseja acessar uma informação, como uma foto tirada por seu drone meses atrás, ele realiza uma requisição para a API. Essa requisição passa primeiro pelo Gateway/Kong, que funciona como a porta de entrada da aplicação. Ele valida o acesso, controla o tráfego, aplica regras de segurança e encaminha a requisição para o serviço interno correto.

### API REST e serviços internos

A API REST é responsável por processar a regra de negócio da aplicação. Ela verifica informações como usuário, permissões, drones vinculados e metadados dos arquivos. Caso o usuário tenha permissão, a API pode acessar o Storage S3 para recuperar o arquivo solicitado.

### Banco de dados

O banco de dados armazena informações estruturadas da aplicação, como usuários, permissões, cadastro de drones, histórico de operações, logs importantes, histórico de buscas e metadados dos arquivos salvos no S3, como nome, caminho, dono, data de envio e permissões de acesso.

### Storage S3 e camada de proteção

O Storage S3 é utilizado para armazenar arquivos grandes, como imagens, vídeos, documentos e registros gerados pelos drones. Dessa forma, esses arquivos não ficam armazenados diretamente no banco de dados principal, evitando sobrecarga e melhorando a escalabilidade da aplicação.

O acesso ao S3 deve ser protegido por uma camada intermediária, como a própria API, proxy ou Gateway/Kong. Isso evita que usuários acessem diretamente arquivos privados ou dados pertencentes a outros usuários.

### Prometheus e Grafana

O Prometheus é responsável por coletar métricas da aplicação, como quantidade de erros, respostas 500, uso de CPU e memória, tempo de resposta e status dos serviços.

O Grafana utiliza essas métricas para exibir dashboards, gráficos e alertas, facilitando o acompanhamento da saúde da API e dos serviços.

### Docker e containers

Os containers Docker ajudam a isolar os serviços da arquitetura, permitindo que cada componente rode em um ambiente controlado e padronizado. Isso facilita a implantação, manutenção e escalabilidade da aplicação.

## Parte 2 — API RESTful com CRUD de Usuários

Nesta etapa foi desenvolvida uma API RESTful utilizando Python com FastAPI.

O objetivo foi criar um CRUD de usuários com autenticação JWT, validação de dados e persistência em banco SQLite.

### Tecnologias usadas nesta etapa

- FastAPI para criação da API REST;
- SQLAlchemy para comunicação com o banco de dados;
- SQLite como banco local;
- Pydantic para validação dos dados de entrada e saída;
- Passlib e bcrypt para hash de senha;
- Python-jose para criação e validação de tokens JWT.

### Estrutura criada

A aplicação foi organizada em camadas para separar responsabilidades:

- `app/main.py`: ponto de entrada da aplicação;
- `app/database.py`: configuração da conexão com o banco SQLite;
- `app/models.py`: model SQLAlchemy da tabela de usuários;
- `app/schemas.py`: schemas Pydantic para validação;
- `app/security.py`: funções de hash de senha e JWT;
- `app/routes/users.py`: rotas de CRUD de usuários;
- `app/routes/auth.py`: rota de autenticação/login.

Essa separação evita concentrar toda a lógica em um único arquivo e facilita manutenção, leitura e evolução do projeto.

### Banco de dados

Foi utilizado SQLite por ser um banco simples, leve e adequado para testes técnicos.

A configuração do banco ficou centralizada no arquivo `app/database.py`.

Esse arquivo cria a conexão com o banco, define a sessão de acesso e disponibiliza uma função `get_db()` para ser usada nas rotas da API.

Centralizar a conexão no `database.py` evita repetição de código nas rotas e facilita manutenção. Caso futuramente fosse necessário trocar SQLite por PostgreSQL, por exemplo, a alteração ficaria concentrada em um ponto principal do projeto.

### Model de usuário

A tabela de usuários foi definida no arquivo `app/models.py` usando SQLAlchemy.

A tabela possui os seguintes campos:

- `id`: identificador único do usuário;
- `name`: nome do usuário;
- `email`: e-mail único usado também no login;
- `hashed_password`: hash da senha do usuário;
- `is_active`: indica se o usuário está ativo;
- `created_at`: data de criação do registro.

A senha original não é armazenada no banco. Antes de salvar, ela é transformada em hash usando bcrypt.

Isso é importante porque, caso o banco de dados seja vazado, a senha em texto puro do usuário não ficará exposta.

### Schemas Pydantic

Os schemas foram definidos no arquivo `app/schemas.py`.

Eles são usados para validar os dados que entram e controlar os dados que saem da API.

Foram criados schemas como:

- `UserCreate`: usado para criação de usuário;
- `UserUpdate`: usado para atualização de usuário;
- `UserResponse`: usado para retornar os dados do usuário;
- `Token`: usado para retornar o token JWT no login;
- `TokenData`: usado para representar dados extraídos do token.

O `UserResponse` não retorna `password` nem `hashed_password`, pois esses dados são sensíveis. Mesmo que o hash não seja a senha em texto puro, ele não deve ser exposto nas respostas da API.

### Rotas implementadas

Foram implementadas as seguintes rotas:

```text
POST   /users/
GET    /users/
GET    /users/{user_id}
PUT    /users/{user_id}
DELETE /users/{user_id}
POST   /auth/login
```

A rota `POST /users/` permite criar um novo usuário.

A rota `GET /users/` lista os usuários cadastrados.

A rota `GET /users/{user_id}` busca um usuário específico pelo ID.

A rota `PUT /users/{user_id}` atualiza os dados de um usuário.

A rota `DELETE /users/{user_id}` desativa um usuário.

A rota `POST /auth/login` autentica o usuário e retorna um token JWT.

### Soft delete

Na rota `DELETE /users/{user_id}`, foi utilizado soft delete.

Em vez de apagar definitivamente o usuário do banco, o campo `is_active` é alterado para `false`.

Essa decisão preserva o histórico e permite que o usuário seja reativado futuramente, se necessário.

### Autenticação JWT

A autenticação foi feita com JWT.

O fluxo funciona da seguinte forma:

1. O usuário cria uma conta em `POST /users/`;
2. O usuário faz login em `POST /auth/login`;
3. A API valida e-mail e senha;
4. Se os dados estiverem corretos, a API gera um token JWT;
5. O usuário envia esse token nas próximas requisições protegidas;
6. A API valida o token antes de liberar o acesso.

O token deve ser enviado no cabeçalho da requisição no formato:

```text
Authorization: Bearer <token>
```

### Rotas públicas e protegidas

As rotas públicas são:

```text
GET  /
POST /users/
POST /auth/login
```

A rota de criação de usuário é pública porque o usuário ainda não possui token antes de se cadastrar.

As rotas protegidas são:

```text
GET    /users/
GET    /users/{user_id}
PUT    /users/{user_id}
DELETE /users/{user_id}
POST   /predict/
```

Essas rotas exigem um token JWT válido.

### Criação automática das tabelas

Para simplificar a execução da prova técnica, as tabelas são criadas automaticamente na inicialização da aplicação com:

```python
Base.metadata.create_all(bind=engine)
```

Em um ambiente de produção, o ideal seria utilizar migrations com Alembic.

## Parte 3 — Integração com Modelo de IA

Nesta etapa foi criada uma integração simples entre a API FastAPI e um modelo de Machine Learning treinado previamente.

O objetivo foi demonstrar o fluxo básico de uso de um modelo de IA em uma API backend.

O fluxo implementado foi:

1. treinar um modelo com dados numéricos;
2. salvar o modelo treinado em arquivo;
3. carregar o modelo dentro da aplicação;
4. receber dados pela API;
5. validar os dados com Pydantic;
6. executar a predição;
7. retornar o resultado em JSON.

### Modelo utilizado

Foi utilizado um modelo simples de regressão linear com `scikit-learn`.

A escolha por regressão linear foi feita porque a prova pede uma predição numérica, e esse tipo de modelo é adequado para demonstrar o fluxo básico de Machine Learning sem adicionar complexidade desnecessária.

O modelo foi treinado no arquivo:

```text
train_model.py
```

Após o treinamento, ele é salvo em:

```text
model/model.joblib
```

O uso do `joblib` permite salvar o modelo treinado em arquivo e reutilizá-lo posteriormente pela API, sem precisar treinar o modelo novamente a cada requisição.

### Separação entre treinamento e predição

O treinamento foi mantido separado da API.

Isso significa que o arquivo `train_model.py` é responsável por treinar e salvar o modelo, enquanto a API apenas carrega o modelo já treinado para fazer previsões.

Essa separação melhora a organização da aplicação e evita processamento desnecessário durante as chamadas da rota `/predict`.

### Serviço de predição

A lógica de carregamento e execução do modelo foi colocada em:

```text
app/services/prediction_service.py
```

Esse service é responsável por:

- verificar se o arquivo do modelo existe;
- carregar o modelo com `joblib`;
- transformar os dados recebidos no formato esperado pelo `scikit-learn`;
- executar a predição;
- retornar o resultado numérico.

O modelo espera os dados no formato bidimensional, por exemplo:

```python
[[feature_1, feature_2, feature_3]]
```

Esse formato representa uma lista de amostras, onde cada amostra possui suas características.

Mesmo quando é feita apenas uma predição, o modelo espera receber os dados como uma lista contendo uma amostra.

### Schemas da predição

Foram criados schemas Pydantic para entrada e saída da rota de predição.

O schema `PredictionRequest` recebe três valores numéricos:

- `feature_1`;
- `feature_2`;
- `feature_3`.

O schema `PredictionResponse` retorna o valor previsto pelo modelo.

Essa validação evita que dados inválidos cheguem até o modelo, como campos ausentes, textos no lugar de números ou formatos inesperados.

### Rota de predição

A rota criada foi:

```text
POST /predict/
```

Essa rota recebe três valores numéricos:

```json
{
  "feature_1": 1,
  "feature_2": 10,
  "feature_3": 100
}
```

E retorna uma previsão:

```json
{
  "prediction": 15.0
}
```

### Segurança da rota

A rota `/predict/` foi protegida com autenticação JWT.

Isso significa que o usuário precisa primeiro criar uma conta, realizar login, obter um token JWT e enviar esse token no cabeçalho da requisição para conseguir acessar a predição.

Exemplo de cabeçalho:

```text
Authorization: Bearer <token>
```

Essa decisão foi tomada porque a predição representa uma funcionalidade interna da API e não deve ficar aberta para usuários não autenticados.

### Observação sobre o modelo

O modelo utilizado é simples e fictício.

O foco desta etapa não foi criar um modelo de Machine Learning complexo, mas demonstrar a integração entre uma API backend e um modelo treinado.

Em um projeto real, o modelo seria treinado com dados reais, teria métricas de avaliação e poderia passar por validações adicionais antes de ser utilizado em produção.

## Parte 4 — Docker e Docker Compose

Nesta etapa foram criados os arquivos necessários para executar a aplicação em container utilizando Docker.

Foram criados:

- `Dockerfile`;
- `docker-compose.yml`.

O objetivo foi facilitar a execução da aplicação em outro ambiente, sem exigir que a avaliadora configure manualmente todas as dependências do projeto.

### Dockerfile

O `Dockerfile` define como a imagem da API deve ser construída.

Ele utiliza uma imagem base do Python, define o diretório de trabalho, copia o arquivo de dependências, instala as bibliotecas necessárias, copia o restante do código e define o comando para iniciar a API com Uvicorn.

O fluxo principal do Dockerfile é:

1. usar uma imagem base com Python;
2. definir o diretório de trabalho dentro do container;
3. copiar o `requirements.txt`;
4. instalar as dependências;
5. copiar o código do projeto;
6. expor a porta `8000`;
7. iniciar a API com Uvicorn.

Copiar o `requirements.txt` antes do restante do código ajuda a aproveitar o cache do Docker.

Como as dependências mudam com menos frequência do que o código da aplicação, o Docker não precisa reinstalar tudo a cada pequena alteração nos arquivos Python, tornando o build mais eficiente.

### Docker Compose

O `docker-compose.yml` foi criado para facilitar a execução da aplicação com um único comando.

Como o projeto usa SQLite, não foi necessário criar um container separado para banco de dados.

O Docker Compose configura:

- o serviço da API;
- o build usando o Dockerfile;
- o mapeamento da porta `8000`;
- um volume para persistir os dados do SQLite;
- a variável de ambiente `DATABASE_URL`.

### Variável de ambiente DATABASE_URL

O arquivo `app/database.py` foi ajustado para ler a variável de ambiente `DATABASE_URL`.

Com isso, a aplicação pode usar uma configuração local durante o desenvolvimento e outra dentro do Docker, sem alterar o código.

Localmente, a aplicação pode usar:

```text
sqlite:///./app.db
```

Dentro do Docker, a aplicação pode usar:

```text
sqlite:///./data/app.db
```

Essa abordagem deixa o projeto mais flexível e facilita adaptações futuras.

### Volume para SQLite

No Docker Compose, foi configurado um volume para persistir o banco SQLite.

Isso evita que os dados sejam perdidos quando o container for recriado.

Como o SQLite funciona como um arquivo local, o volume garante que esse arquivo continue armazenado mesmo após reiniciar a aplicação.

### Como rodar com Docker

Para subir a aplicação com Docker Compose, basta executar:

```bash
docker compose up --build
```

Após subir a aplicação, a documentação Swagger pode ser acessada em:

```text
http://127.0.0.1:8000/docs
```

Para encerrar os containers, o comando é:

```bash
docker compose down
```

### Observações técnicas

O Docker foi utilizado para padronizar o ambiente de execução da API.

Isso reduz problemas relacionados a diferenças entre máquinas e facilita a avaliação da prova técnica.

Em um ambiente de produção, seria recomendado também configurar variáveis sensíveis fora do código, como `SECRET_KEY`, credenciais e outros dados de ambiente.